# ParsingAI — Colab 版
依序執行每個 Cell。

In [ ]:
# ── Cell 1: Clone repo & 安裝套件 ─────────────────────────────────────────────
!git clone https://github.com/410773004/parsingAI.git /content/parsingAI -q
!pip install ollama tiktoken -q

import sys
sys.path.insert(0, '/content/parsingAI')
print('✓ 完成')

In [ ]:
# ── Cell 2: 安裝 Ollama & 拉模型（約 2.5GB，需等幾分鐘）────────────────────────
import sys
if '/content/parsingAI' not in sys.path:
    sys.path.insert(0, '/content/parsingAI')

!apt-get install -y zstd pciutils -q
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

from settings import config
print(f'拉取模型：{config.MODEL}')
!ollama pull {config.MODEL}
print('✓ 完成')

In [ ]:
# ── Cell 3: 上傳 log 資料夾的 ZIP 檔 ──────────────────────────────────────────
import sys
if '/content/parsingAI' not in sys.path:
    sys.path.insert(0, '/content/parsingAI')

from google.colab import files
import zipfile
from pathlib import Path

uploaded = files.upload()  # 選擇 ZIP 檔
zip_name = list(uploaded.keys())[0]
folder_name = Path(zip_name).stem

extract_dir = Path(f'/content/{folder_name}')
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(zip_name, 'r') as z:
    members = z.namelist()
    top_dirs = {m.split('/')[0] for m in members if '/' in m}
    if len(top_dirs) == 1 and list(top_dirs)[0] == folder_name:
        z.extractall('/content/')
    else:
        z.extractall(extract_dir)

log_files = list(extract_dir.glob('*.log'))
print(f'解壓完成：{extract_dir}')
print(f'找到 {len(log_files)} 個 .log 檔案')

In [ ]:
# ── Cell 4: 執行分析 ───────────────────────────────────────────────────────────
import sys, subprocess, time, importlib
if '/content/parsingAI' not in sys.path:
    sys.path.insert(0, '/content/parsingAI')

# 確保 Ollama server 在跑
import httpx
try:
    httpx.get('http://127.0.0.1:11434', timeout=2)
except Exception:
    print('啟動 Ollama server...')
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

# 修改下面這行描述問題
issue = "請輸入問題描述"

from tqdm.notebook import tqdm
from IPython.display import display, clear_output
import ipywidgets as widgets
import services.analyzer as _analyzer
importlib.reload(_analyzer)
from services.analyzer import analyze

# 進度條
STAGES = ['偵測 project', '解析 log', '分析 Event Flow', '分析溫度', '抽取 metadata', '計算 token', 'LLM 分析中']
pbar = tqdm(total=len(STAGES), bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}]')

def _stage_cb(stage):
    pbar.set_description(stage)
    if any(stage.startswith(s) for s in STAGES):
        pbar.update(1)

# LLM 串流輸出
llm_out = widgets.Output()
display(llm_out)
_llm_text = ""

def _on_token(token):
    global _llm_text
    _llm_text += token
    with llm_out:
        clear_output(wait=True)
        print(_llm_text, end="", flush=True)

_analyzer.set_stage_callback(_stage_cb)
result = analyze(extract_dir, issue, folder_name, on_token=_on_token)
pbar.close()

print(f"\nProject    : {result['project']}")
print(f"FW Version : {result['fw_version']}")
print(f"Serial     : {result['serial']}")
print(f"Token 數   : {result['token_count']}")

In [ ]:
# ── Cell 5: 儲存結果到檔案（選用）────────────────────────────────────────────
output_path = Path(f'/content/{folder_name}_result.txt')
output_path.write_text(
    f"Project    : {result['project']}\n"
    f"FW Version : {result['fw_version']}\n"
    f"Serial     : {result['serial']}\n"
    f"Token 數   : {result['token_count']}\n"
    f"{'=' * 80}\n"
    f"{result['llm_output']}\n"
    f"{'=' * 80}\n"
    f"{result['cleaned_log']}",
    encoding='utf-8'
)

files.download(str(output_path))
print(f'已下載：{output_path.name}')

In [ ]:
# ── Cell 7: 直接餵 LLM ────────────────────────────────────────────────────────
import sys, subprocess, time
if '/content/parsingAI' not in sys.path:
    sys.path.insert(0, '/content/parsingAI')

# 確保 Ollama server 在跑
import httpx
try:
    httpx.get('http://127.0.0.1:11434', timeout=2)
except Exception:
    print('啟動 Ollama server...')
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)

# 修改以下三行
issue      = "請輸入問題描述"
project    = "PJ1"   # PJ1 or ER3
fw_version = ""

from IPython.display import display, clear_output
import ipywidgets as widgets
import ollama
from settings import config, prompts

user_prompt = prompts.USER_TEMPLATE.format(
    model=project, fw_version=fw_version, issue=issue, log_text=cleaned_log
)
messages = [
    {"role": "system", "content": prompts.SYSTEM},
    {"role": "user",   "content": user_prompt},
]
options = {
    "temperature":    config.TEMPERATURE,
    "top_p":          config.TOP_P,
    "top_k":          config.TOP_K,
    "repeat_penalty": config.REPEAT_PENALTY,
    "num_ctx":        config.NUM_CTX,
}

llm_out = widgets.Output()
display(llm_out)
full_text = ""

for chunk in ollama.chat(model=config.MODEL, messages=messages, options=options, stream=True):
    token = chunk.get("message", {}).get("content", "")
    full_text += token
    with llm_out:
        clear_output(wait=True)
        print(full_text, end="", flush=True)

In [ ]:
# ── Cell 6: 上傳處理過的 log txt ──────────────────────────────────────────────
import sys, subprocess, time
if '/content/parsingAI' not in sys.path:
    sys.path.insert(0, '/content/parsingAI')

from google.colab import files
from pathlib import Path

uploaded = files.upload()
log_filename = list(uploaded.keys())[0]
cleaned_log = Path(log_filename).read_text(encoding='utf-8', errors='ignore')

from parsers.compress import count_tokens
print(f'檔案：{log_filename}')
print(f'Token 數：{count_tokens(cleaned_log)}')

---
## 直接上傳處理過的 log（跳過 pipeline）
已經有處理好的 log txt 的話，用下面兩個 Cell 直接餵 LLM。